In [1]:
# List all files under data/rawdomains/* 

from pathlib import Path
import os
import pandas as pd

# Locate repo root 
CWD = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError("Could not find repo root (no parent directory contains a /data folder).")

REPO_ROOT = find_repo_root(CWD)
RAWDOMAINS_DIR = REPO_ROOT / "data" / "rawdomains"

print("Notebook CWD:", CWD)
print("Repo root:", REPO_ROOT)
print("Rawdomains dir:", RAWDOMAINS_DIR)

if not RAWDOMAINS_DIR.exists():
    raise FileNotFoundError(f"Missing folder: {RAWDOMAINS_DIR}")

# Collect files
rows = []
for path in RAWDOMAINS_DIR.rglob("*"):
    if path.is_file():
        rel = path.relative_to(REPO_ROOT)
        rows.append({
            "relative_path": str(rel),
            "domain_folder": path.relative_to(RAWDOMAINS_DIR).parts[0] if len(path.relative_to(RAWDOMAINS_DIR).parts) else "",
            "filename": path.name,
            "ext": path.suffix.lower(),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 4),
        })

files_df = pd.DataFrame(rows).sort_values(["domain_folder", "relative_path"]).reset_index(drop=True)

def print_tree(base: Path):
    print("\n data/rawdomains tree")
    for root, dirs, files in os.walk(base):
        rel_root = Path(root).relative_to(base)
        indent = "  " * len(rel_root.parts)
        folder_name = "." if str(rel_root) == "." else rel_root.name
        print(f"{indent}{folder_name}/")
        for f in sorted(files):
            fp = Path(root) / f
            mb = fp.stat().st_size / (1024 * 1024)
            print(f"{indent}  - {f} ({mb:.3f} MB)")

print_tree(RAWDOMAINS_DIR)

# Summary by Domain
print("\n Counts by domain folder")
if len(files_df) == 0:
    print("No files found under data/rawdomains.")
else:
    display(files_df.groupby(["domain_folder", "ext"]).size().reset_index(name="count").sort_values(["domain_folder","count"], ascending=[True, False]))

# Show full list (filter/sort in notebook UI)
display(files_df)

# Save inventory CSV next to other processed data
out_path = REPO_ROOT / "data" / "processed" / "rawdomains_file_inventory.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
files_df.to_csv(out_path, index=False)
print(f"\nSaved inventory to: {out_path}")

Notebook CWD: /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks
Repo root: /Users/laurenvo/Documents/github/youth_opportunity_index
Rawdomains dir: /Users/laurenvo/Documents/github/youth_opportunity_index/data/rawdomains

 data/rawdomains tree
./
  health/
    - PLACES__Local_Data_for_Better_Health,_Census_Tract_Data,_2025_release_20260311.csv (8.121 MB)
    ACSST5Y2024.S2701_2026-03-11T050431/
      - ACSST5Y2024.S2701-Column-Metadata.csv (0.094 MB)
      - ACSST5Y2024.S2701-Data.csv (2.739 MB)
      - ACSST5Y2024.S2701-Table-Notes.txt (0.011 MB)
    ACSST5Y2024.S1810_2026-03-11T050449/
      - ACSST5Y2024.S1810-Column-Metadata.csv (0.073 MB)
      - ACSST5Y2024.S1810-Data.csv (1.801 MB)
      - ACSST5Y2024.S1810-Table-Notes.txt (0.011 MB)
  economic/
    ACSDT5Y2024.B01003_2026-03-11T063504/
      - ACSDT5Y2024.B01003-Column-Metadata.csv (0.000 MB)
      - ACSDT5Y2024.B01003-Data.csv (0.062 MB)
      - ACSDT5Y2024.B01003-Table-Notes.txt (0.011 MB)
    ACSDT5Y2024.B19

,domain_folder,ext,count
0,economic,.csv,10
1,economic,.txt,5
2,education,.csv,6
3,education,.txt,3
4,health,.csv,5
5,health,.txt,2
6,housing,.csv,8
7,housing,.txt,4
9,mobility,.csv,7
14,mobility,.txt,3


,relative_path,domain_folder,filename,ext,size_mb
0,data/rawdomains/economic/ACSDT5Y2024.B01003_20...,economic,ACSDT5Y2024.B01003-Column-Metadata.csv,.csv,0.0001
1,data/rawdomains/economic/ACSDT5Y2024.B01003_20...,economic,ACSDT5Y2024.B01003-Data.csv,.csv,0.0623
2,data/rawdomains/economic/ACSDT5Y2024.B01003_20...,economic,ACSDT5Y2024.B01003-Table-Notes.txt,.txt,0.0108
3,data/rawdomains/economic/ACSDT5Y2024.B19013_20...,economic,ACSDT5Y2024.B19013-Column-Metadata.csv,.csv,0.0003
4,data/rawdomains/economic/ACSDT5Y2024.B19013_20...,economic,ACSDT5Y2024.B19013-Data.csv,.csv,0.0648
...,...,...,...,...,...
75,data/rawdomains/safety/calenviroscreen40shpf20...,safety,CES4 Final Shapefile.shp,.shp,8.0562
76,data/rawdomains/safety/calenviroscreen40shpf20...,safety,CES4 Final Shapefile.shp.xml,.xml,0.0698
77,data/rawdomains/safety/calenviroscreen40shpf20...,safety,CES4 Final Shapefile.shx,.shx,0.0614
78,data/rawdomains/youth/services_master.csv,youth,services_master.csv,.csv,4.6610



Saved inventory to: /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/rawdomains_file_inventory.csv


In [2]:
pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import re

# --- paths: find repo root that contains /data ---
REPO_ROOT = Path.cwd()
for p in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (p / "data").exists():
        REPO_ROOT = p
        break

shp = REPO_ROOT / "data/rawdomains/safety/calenviroscreen40shpf2021shp/CES4 Final Shapefile.shp"
out_dir = REPO_ROOT / "data/processed/boundaries"
out_dir.mkdir(parents=True, exist_ok=True)

assert shp.exists(), f"Could not find shapefile at: {shp}"

gdf = gpd.read_file(shp)

# --- find tract id column ---
tract_col = None
for cand in ["Tract", "TRACT", "GEOID", "geoid", "Census Tract", "census_tract", "FIPS", "fips"]:
    if cand in gdf.columns:
        tract_col = cand
        break
if tract_col is None:
    for c in gdf.columns:
        if "tract" in c.lower() or "geoid" in c.lower():
            tract_col = c
            break
assert tract_col is not None, f"Could not find tract id column. Columns: {list(gdf.columns)}"

print("Using tract_col =", tract_col)
print("Sample tract_col values:", gdf[tract_col].head(10).tolist())

def geoid11_from_ces_tract(x):
    """
    CalEnviroScreen shapefile 'Tract' often looks like: 6083002103.0
    We want: 06083002103 (11-digit GEOID = state+county+tract)
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]              # remove float artifact
    digits = re.sub(r"\D", "", s)
    if len(digits) == 10:
        return digits.zfill(11) # add leading 0 -> 11 digits
    if len(digits) == 11:
        return digits
    return None

# Compute tract_geoid ONCE (don't overwrite later)
gdf["tract_geoid"] = gdf[tract_col].map(geoid11_from_ces_tract).astype("string")

# Filter to San Diego County (06073)
gdf = gdf[gdf["tract_geoid"].notna()].copy()
gdf = gdf[gdf["tract_geoid"].str.startswith("06073")].copy()

print("SD tracts:", len(gdf))
print("Example computed tract_geoid:", gdf["tract_geoid"].head(10).tolist())
print("tract_geoid length counts:\n", gdf["tract_geoid"].astype(str).str.len().value_counts())

# Write a clean boundary file with properties Folium can use
out_path = out_dir / "sd_tracts.geojson"
gdf[["tract_geoid", "geometry"]].to_file(out_path, driver="GeoJSON")
print("Wrote:", out_path)

Using tract_col = Tract
Sample tract_col values: [6083002103.0, 6083002402.0, 6083002102.0, 6083002010.0, 6083002009.0, 6083002008.0, 6083002007.0, 6011000500.0, 6011000200.0, 6011000400.0]
SD tracts: 627
Example computed tract_geoid: ['06073002702', '06073002710', '06073002707', '06073002705', '06073002202', '06073002601', '06073008352', '06073008353', '06073008354', '06073008355']
tract_geoid length counts:
 tract_geoid
11    627
Name: count, dtype: int64
Wrote: /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/boundaries/sd_tracts.geojson


In [7]:
df = pd.read_csv("../data/processed/yoi/yoi_components.csv", dtype={"tract_geoid": str})
print(df.columns.tolist())
print(df["total_population"].isna().sum())
print(df.loc[df["total_population"].isna(), ["tract_geoid"]].head(20))

['tract_geoid', 'economic_score', 'economic_coverage', 'education_score', 'education_coverage', 'health_score', 'health_coverage', 'housing_score', 'housing_coverage', 'safety_env_score', 'safety_env_coverage', 'mobility_connectivity_score', 'mobility_connectivity_coverage', 'youth_supports_score', 'youth_supports_coverage', 'yoi_raw_0_1', 'yoi_0_100', 'total_population', 'crime_rate_per_1k', 'service_count', 'services_per_10k', 'youth_services_per_10k', 'mh_services_per_10k']
92
     tract_geoid
8    06073008354
12   06073018100
13   06073018200
14   06073018300
15   06073019106
20   06073019207
38   06073010200
50   06073018507
55   06073018513
56   06073018514
57   06073019302
81   06073004100
108  06073019806
113  06073020013
114  06073020014
116  06073020016
118  06073018614
125  06073021100
128  06073001300
129  06073001200


In [8]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "data").exists():
    root = root.parent

# final output
df = pd.read_csv(root / "data" / "processed" / "yoi" / "yoi_components.csv",
                 dtype={"tract_geoid": str})

print("NaNs in final total_population:", df["total_population"].isna().sum())
missing_final = set(df.loc[df["total_population"].isna(), "tract_geoid"].astype(str))
print("Example missing in final:", sorted(list(missing_final))[:20])

# raw B01003 file
b01003_path = next((root / "data" / "rawdomains" / "economic").glob("ACSDT5Y2024.B01003_*/*-Data.csv"))
pop = pd.read_csv(b01003_path, dtype=str)
print("B01003 columns:", pop.columns.tolist())

# pick the GEO column the same way your loader does
geo_col = None
for cand in ["tract_geoid", "GEOID", "GEO_ID", "geo_id", "LocationName", "locationname"]:
    if cand in pop.columns:
        geo_col = cand
        break
if geo_col is None:
    geo_col = pop.columns[0]

def digits_only(x):
    import re
    return re.sub(r"\D", "", str(x))

def geoid11_from_any(x):
    d = digits_only(x)
    return d[-11:] if len(d) >= 11 else None

pop["tract_geoid"] = pop[geo_col].apply(geoid11_from_any)
pop = pop[pop["tract_geoid"].notna()].copy()
pop = pop[pop["tract_geoid"].str.startswith("06073")].copy()

print("Chosen B01003 geo column:", geo_col)
print("Unique B01003 SD tracts:", pop["tract_geoid"].nunique())

missing_from_b01003 = sorted(list(missing_final - set(pop["tract_geoid"].astype(str))))
print("Missing from B01003 after parsing:", missing_from_b01003[:50])

NaNs in final total_population: 92
Example missing in final: ['06073000300', '06073000900', '06073001200', '06073001300', '06073001800', '06073004100', '06073005100', '06073005200', '06073005300', '06073005400', '06073005600', '06073005800', '06073007301', '06073007400', '06073007600', '06073007903', '06073008200', '06073008329', '06073008333', '06073008335']
B01003 columns: ['GEO_ID', 'NAME', 'B01003_001E', 'B01003_001M', 'Unnamed: 4']
Chosen B01003 geo column: GEO_ID
Unique B01003 SD tracts: 737
Missing from B01003 after parsing: ['06073000300', '06073000900', '06073001200', '06073001300', '06073001800', '06073004100', '06073005100', '06073005200', '06073005300', '06073005400', '06073005600', '06073005800', '06073007301', '06073007400', '06073007600', '06073007903', '06073008200', '06073008329', '06073008333', '06073008335', '06073008340', '06073008341', '06073008354', '06073009106', '06073009202', '06073009304', '06073010014', '06073010200', '06073013310', '06073013311', '0607301331